# 01 — Camera Geometry and the Pinhole Model

**Objective:** Understand and visualise the coordinate systems and mathematical transforms that map a 3-D world point onto a 2-D pixel — and understand where distortion enters.

**Prerequisites:** Notebook `00_problem_and_setup.ipynb` must pass its smoke test.

---

## 1.1  Coordinate systems in camera geometry

Four coordinate systems are involved:

```
World (3-D)  →  Camera (3-D)  →  Image plane (2-D)  →  Pixel (2-D)
    [R|t]              K                  pixels
```

| System | Description |
|---|---|
| **World** | Arbitrary 3-D frame where object points are defined |
| **Camera** | Origin at optical centre, Z points forward along optical axis |
| **Image plane** | 2-D plane at distance f (focal length) from optical centre |
| **Pixel** | Discrete integer grid with (0,0) at top-left corner |

## 1.2  The projection equation

A world point **X** = [X, Y, Z]ᵀ maps to a pixel **x** = [u, v]ᵀ via:

$$\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \mathbf{K} [\mathbf{R} | \mathbf{t}] \begin{bmatrix} X \\ Y \\ Z \\ 1 \end{bmatrix}$$

where:

$$\mathbf{K} = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

- $(f_x, f_y)$ — focal lengths in **pixels** (not millimetres).
- $(c_x, c_y)$ — principal point: where the optical axis pierces the image plane.
- $[\mathbf{R}|\mathbf{t}]$ — 3×4 extrinsic matrix (rotation + translation from world to camera).

## 1.3  Where lens distortion appears

The pinhole model is an idealisation. Real lenses bend light unevenly, causing:

- **Barrel distortion** (k₁ < 0): edges bow outward — common in wide-angle and phone cameras.
- **Pincushion distortion** (k₁ > 0): edges bow inward — common in telephoto lenses.
- **Tangential distortion** (p₁, p₂): lens is not perfectly aligned with the sensor.

The distortion model (Brown-Conrady):

$$x_d = x_u (1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + 2p_1 x_u y_u + p_2(r^2 + 2x_u^2)$$
$$y_d = y_u (1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + p_1(r^2 + 2y_u^2) + 2p_2 x_u y_u$$

where $r^2 = x_u^2 + y_u^2$ and $(x_u, y_u)$ are **normalised** (undistorted) camera coordinates.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Make calibvision importable
SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

print('Imports OK')

## 1.4  Numerical example: projecting a 3-D point

Let's project a single world point step-by-step to see each coordinate system.

In [ ]:
# ── Example: project one 3-D point through K and [R|t] ───────────────────────

# Synthetic camera intrinsics (typical values for a 1280×720 webcam)
fx, fy = 800.0, 800.0   # focal lengths in pixels
cx, cy = 640.0, 360.0   # principal point

K = np.array([[fx,  0, cx],
              [ 0, fy, cy],
              [ 0,  0,  1]], dtype=np.float64)

print('Intrinsic matrix K:')
print(K)
print(f'\nFocal lengths: fx={fx} px, fy={fy} px')
print(f'Principal point: cx={cx}, cy={cy}')

# Identity extrinsic (camera IS the world frame — simplest case)
R = np.eye(3)
t = np.array([[0.0], [0.0], [0.0]])
Rt = np.hstack([R, t])   # 3×4

# A world point at [X=0.2, Y=0.1, Z=1.5] metres in front of the camera
X_world = np.array([[0.2], [0.1], [1.5], [1.0]])  # homogeneous

# Step 1 — camera coordinates
X_cam = Rt @ X_world
print(f'\n3-D camera coords:  X_c={X_cam[0,0]:.4f}  Y_c={X_cam[1,0]:.4f}  Z_c={X_cam[2,0]:.4f}')

# Step 2 — normalised image coords (divide by Z)
x_n = X_cam[0, 0] / X_cam[2, 0]
y_n = X_cam[1, 0] / X_cam[2, 0]
print(f'Normalised image:   x_n={x_n:.4f}  y_n={y_n:.4f}')

# Step 3 — pixel coordinates via K
x_hom = K @ X_cam
u = x_hom[0, 0] / x_hom[2, 0]
v = x_hom[1, 0] / x_hom[2, 0]
print(f'Pixel coordinates:  u={u:.2f}  v={v:.2f}  (image size 1280×720)')

## 1.5  Visualising lens distortion

The following visualisation simulates how different distortion coefficients bend a regular grid.

In [ ]:
import cv2

def distort_grid(k1: float, k2: float = 0.0, p1: float = 0.0, p2: float = 0.0,
                 img_size: tuple = (640, 480)) -> np.ndarray:
    """
    Apply distortion to a regular dot-grid image and return the warped result.
    
    This isolates the visual effect of each distortion coefficient.
    """
    w, h = img_size
    
    # Synthetic camera matrix centred in the image
    f = w * 0.85   # focal length ≈ 85% of width
    K_syn = np.array([[f, 0, w/2], [0, f, h/2], [0, 0, 1]], dtype=np.float64)
    D = np.array([[k1, k2, p1, p2, 0.0]])
    
    # Build a regular grid of undistorted pixel coordinates
    xs = np.linspace(20, w - 20, 18)
    ys = np.linspace(20, h - 20, 14)
    grid_x, grid_y = np.meshgrid(xs, ys)
    src_pts = np.stack([grid_x.ravel(), grid_y.ravel()], axis=-1).astype(np.float32)
    
    # Distort the ideal points → simulates where they'd appear in a distorted image
    distorted = cv2.undistortPoints(src_pts.reshape(-1, 1, 2), K_syn, D)
    distorted = cv2.convertPointsFromHomogeneous(distorted).reshape(-1, 2)
    # Back to pixel space
    distorted[:, 0] = distorted[:, 0] * K_syn[0, 0] + K_syn[0, 2]
    distorted[:, 1] = distorted[:, 1] * K_syn[1, 1] + K_syn[1, 2]
    
    # Draw on a blank canvas
    canvas = np.full((h, w, 3), 240, dtype=np.uint8)
    for pt in distorted:
        cx_pt, cy_pt = int(pt[0]), int(pt[1])
        if 0 <= cx_pt < w and 0 <= cy_pt < h:
            cv2.circle(canvas, (cx_pt, cy_pt), 4, (30, 100, 220), -1)
    
    # Draw original grid (grey dots) for comparison
    for pt in src_pts:
        cv2.circle(canvas, (int(pt[0]), int(pt[1])), 2, (180, 180, 180), -1)
    
    return canvas


configs = [
    {'k1': 0.0,   'title': 'No distortion (ideal)'},
    {'k1': -0.4,  'title': 'Barrel (k₁ = -0.4)'},
    {'k1':  0.4,  'title': 'Pincushion (k₁ = +0.4)'},
    {'k1': -0.2, 'p1': 0.1, 'title': 'Barrel + tangential'},
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, cfg in zip(axes, configs):
    img = distort_grid(
        k1=cfg.get('k1', 0), k2=cfg.get('k2', 0),
        p1=cfg.get('p1', 0), p2=cfg.get('p2', 0)
    )
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(cfg['title'], fontsize=9)
    ax.axis('off')

# Legend
blue_patch = mpatches.Patch(color='#1e64dc', label='Distorted position')
grey_patch = mpatches.Patch(color='#b4b4b4', label='Ideal (undistorted)')
fig.legend(handles=[blue_patch, grey_patch], loc='lower center', ncol=2, fontsize=9)
plt.suptitle('Effect of lens distortion coefficients on a regular grid', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 1.6  Reprojection error — the calibration quality metric

After estimating K and D, we verify calibration quality by **reprojecting** the known 3-D board points back onto each calibration image and measuring how far they land from the detected 2-D corners:

$$\text{RMS} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \|x_i - \hat{x}_i\|^2}$$

- $x_i$ = detected corner pixel
- $\hat{x}_i$ = reprojected corner pixel
- **Typical target: RMS < 0.5–1.0 pixels** for a well-calibrated camera.


In [ ]:
# ── Simulate what reprojection error looks like numerically ───────────────────
np.random.seed(42)

# Hypothetical reprojection errors for 20 calibration images
n_views = 20
true_errors = np.abs(np.random.normal(loc=0.35, scale=0.15, size=n_views))
rms = float(np.sqrt(np.mean(true_errors ** 2)))

print(f'Simulated RMS reprojection error: {rms:.4f} px')
print(f'Min per-view error: {true_errors.min():.4f} px')
print(f'Max per-view error: {true_errors.max():.4f} px')
print(f'\nInterpretation:')
print(f'  < 0.5 px → Excellent calibration')
print(f'  0.5–1.0 px → Acceptable for most applications')
print(f'  > 1.0 px → Investigate: blurry images? bent board? too few views?')

# Import and use the real plotting utility from calibvision
from calibvision.utils import draw_reprojection_error
names = [f'img_{i:02d}.jpg' for i in range(n_views)]
fig = draw_reprojection_error(true_errors.tolist(), names, rms, max_acceptable=1.0)
plt.show()
print('\n✓ draw_reprojection_error utility works correctly')

## Summary

| Concept | Key takeaway |
|---|---|
| Pinhole model | Maps 3-D points to 2-D pixels via K, R, t |
| Intrinsic K | Encodes focal length and principal point — **camera-specific** |
| Distortion D | Brown-Conrady radial + tangential — **lens-specific** |
| Reprojection error | Primary calibration quality metric — target < 1.0 px |
| Barrel / Pincushion | Visualised above — k₁ negative = barrel, positive = pincushion |

## Next step

Open `02_calibration_data_exploration.ipynb` to load real checkerboard images and inspect corner detection results.
